In [35]:
import fitz  # PyMuPDF
import pandas as pd

In [36]:
pdf_path = "test_sample/Ficha_Ponto_Simplificada_André_Luis.pdf"

In [37]:
# pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
doc = fitz.open(pdf_path)
# self.timesheet_data = []

for page in doc:
    tabs = page.find_tables()

    for tab in tabs.tables:
        print(tab.to_pandas())

             Col0      Col1      Col2 SCALA TRANSPORTE E ADMINISTRACAO LTDA. 88.501.093/0001-89     Col4   Col5       Col6             Col7             Col8           Col9  ...            Col16           Col17                                              Col18                  Col19    Col20  Col21                   Col22    Col23  Col24          Col25
0                      None      None  Ficha Ponto Simplificada Período: 21/05/2025 à...            None   None       None             None             None           None  ...             None            None                                               None                   None     None   None                    None     None   None           None
1            None      None      None  Funcionário: ANDRE LUIS DE MORAES DA ROSA CPF:...            None   None       None             None             None           None  ...             None            None                                               None                   None     No

In [38]:
import re
from pandas import DataFrame

# Convert the table to a DataFrame
df_tab = tab.to_pandas()

def reset_column_names(df: DataFrame) -> DataFrame:
    df.columns = [f"Col_{i}" for i in range(df.shape[1])]
    return df

# Define a function to check if a string starts with a date in the format DD/MM/YY
def starts_with_date(val):
    if isinstance(val, str):
        return bool(re.match(r'^\d{2}/\d{2}/\d{2}', val.strip()))
    return False

def drop_unwanted_columns(df: DataFrame) -> DataFrame:
    # Drop all columns that are fully with None
    df = df.dropna(axis=1, how='all')
    return df

# Filter rows where the first column starts with a date
df_tab = reset_column_names(df_tab)
filtered_df = df_tab[df_tab.iloc[:, 0].apply(starts_with_date)].reset_index(drop=True)
df_timesheet = drop_unwanted_columns(filtered_df)


print(df_timesheet)

           Col_0     Col_1  Col_4  Col_5  Col_7  Col_8 Col_10 Col_11 Col_13 Col_14 Col_16 Col_17 Col_19 Col_20 Col_21 Col_22 Col_23 Col_24 Col_25
0   21/05/25 qua  Trabalho  06:24  01:26  08:00  16:59  05:30  12:11  04:48  04:48  01:21  00:42  05:57  03:02  08:59                       03:02
1   22/05/25 qui  Trabalho  10:35  23:42  08:00  11:58  09:09  04:03  07:55  07:55  01:09         02:16  01:42  03:58                       01:42
2   23/05/25 sex  Trabalho  08:05  20:32  08:00  10:56  08:23  05:10  05:46  05:46  01:11  00:20  02:56         02:56                            
3   24/05/25 sáb  Trabalho  08:22  18:02  08:00  08:33  11:50  01:02  07:31  07:31  01:07         00:33         00:33                            
4   25/05/25 dom  Trabalho  07:37  19:07  08:00  10:08  13:35  05:46  04:22  04:22  01:22         02:08         02:08                            
5   26/05/25 seg  Trabalho  05:54  21:17  08:00  14:21  10:47  03:00  11:21  11:21  01:02         06:21         06:21       

In [39]:
column_mapping = {
    'Col_0': 'data',
    'Col_1': 'tipo',
    'Col_4': 'jornada_inicio',
    'Col_5': 'jornada_fim',
    'Col_7': 'jornada_normal',
    'Col_8': 'jornada_diaria',
    'Col_10': 'interjornada',
    'Col_11': 'em_direcao',
    'Col_13': 'total_parado',
    'Col_14': 'sem_direcao',
    'Col_16': 'total_refeicao',
    'Col_17': 'total_repouso',
    'Col_19': 'hora_extra_diaria_diurna',
    'Col_20': 'hora_extra_diaria_noturna',
    'Col_21': 'hora_extra_diaria_total',
    'Col_22': 'hora_extra_dom_fer_diurna',
    'Col_23': 'hora_extra_dom_fer_noturna',
    'Col_24': 'hora_extra_dom_fer_total',
    'Col_25': 'hora_noturna'
}

df_timesheet_renamed = df_timesheet.rename(columns=column_mapping)
print(df_timesheet_renamed)

            data      tipo jornada_inicio jornada_fim jornada_normal jornada_diaria interjornada em_direcao total_parado sem_direcao total_refeicao total_repouso hora_extra_diaria_diurna hora_extra_diaria_noturna hora_extra_diaria_total hora_extra_dom_fer_diurna hora_extra_dom_fer_noturna hora_extra_dom_fer_total hora_noturna
0   21/05/25 qua  Trabalho          06:24       01:26          08:00          16:59        05:30      12:11        04:48       04:48          01:21         00:42                    05:57                     03:02                   08:59                                                                                      03:02
1   22/05/25 qui  Trabalho          10:35       23:42          08:00          11:58        09:09      04:03        07:55       07:55          01:09                                  02:16                     01:42                   03:58                                                                                      01:42
2   23/05/25 sex

In [40]:
# Extract information from the PDF
import re
from datetime import datetime

def extract_timesheet_info(pdf_path):
    doc = fitz.open(pdf_path)
    
    # Variables to store extracted information
    company_name = None
    employee_name = None
    period = None
    daily_data = []
    
    for page in doc:
        # Extract text from the page for company, employee, and period info
        text = page.get_text()
        
        # Extract company name (usually appears at the top)
        # Look for common patterns in Portuguese timesheets
        company_patterns = [
            r'EMPRESA[:\s]*([^\n\r]+)',
            r'RAZÃO SOCIAL[:\s]*([^\n\r]+)',
            r'EMPREGADOR[:\s]*([^\n\r]+)'
        ]
        
        for pattern in company_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match and not company_name:
                company_name = match.group(1).strip()
                break
        
        # Extract employee name
        employee_patterns = [
            r'EMPREGADO[:\s]*([^\n\r]+)',
            r'FUNCIONÁRIO[:\s]*([^\n\r]+)',
            r'NOME[:\s]*([^\n\r]+)',
        ]
        
        for pattern in employee_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match and not employee_name:
                employee_name = match.group(1).strip() if match.group(1) else match.group(0).strip()
                break
        
        # Extract period
        period_patterns = [
            r'PERÍODO[:\s]*([^\n\r]+)',
            r'COMPETÊNCIA[:\s]*([^\n\r]+)',
            r'(\d{2}/\d{4})',  # MM/YYYY format
            r'(\d{2}/\d{2}/\d{4}\s*a\s*\d{2}/\d{2}/\d{4})'  # DD/MM/YYYY a DD/MM/YYYY format
        ]
        
        for pattern in period_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match and not period:
                period = match.group(1).strip()
                break
        
        # Extract table data
        tabs = page.find_tables()
        
        for tab in tabs.tables:
            df = tab.to_pandas()
            
            # Look for tables that contain daily timesheet data
            # Common column patterns in Portuguese timesheets
            if len(df.columns) >= 3:
                # Check if this looks like a daily timesheet table
                df_text = df.astype(str).values.flatten()
                has_dates = any(re.search(r'\d{1,2}/\d{1,2}', str(cell)) for cell in df_text)
                has_times = any(re.search(r'\d{1,2}:\d{2}', str(cell)) for cell in df_text)
                
                if has_dates or has_times:
                    daily_data.append(df)
    
    doc.close()
    
    return company_name, employee_name, period, daily_data

# Extract the information
company_name, employee_name, period, daily_data_list = extract_timesheet_info(pdf_path)

print("=== EXTRACTED INFORMATION ===")
print(f"Company/Employer: {company_name}")
print(f"Employee Name: {employee_name}")
print(f"Period: {period}")
print(f"\nNumber of daily data tables found: {len(daily_data_list)}")

=== EXTRACTED INFORMATION ===
Company/Employer: None
Employee Name: Período: 21/05/2025 à 20/06/2025
Period: 21/05/2025 à 20/06/2025

Number of daily data tables found: 1


In [41]:
# Fix employee name extraction from the raw text
doc = fitz.open(pdf_path)
text = doc[0].get_text()
doc.close()

# Extract employee name more accurately
employee_name_match = re.search(r'ANDRE LUIS DE MORAES DA ROSA', text)
if employee_name_match:
    employee_name = employee_name_match.group(0)
else:
    # Fallback pattern
    employee_name_match = re.search(r'Funcionário:\s*\n?([A-ZÁÊÇÕ\s]+)', text)
    if employee_name_match:
        employee_name = employee_name_match.group(1).strip()

daily_df = df_timesheet_renamed
print("=== FINAL EXTRACTED INFORMATION ===")
print(f"Company/Employer: {company_name}")
print(f"Employee Name: {employee_name}")  
print(f"Period: {period}")
print(f"\n=== SUMMARY OF DAILY TIMESHEET ===")
print(f"Total days in timesheet: {len(daily_df)}")
print(f"Working days: {len(daily_df[daily_df['tipo'] == 'Trabalho'])}")
print(f"Rest days (DSR/Casa): {len(daily_df[daily_df['tipo'] == 'DSR/Casa'])}")
print(f"Holidays (Feriado): {len(daily_df[daily_df['tipo'] == 'Feriado'])}")

print(f"\n=== TIMESHEET DATAFRAME COLUMNS ===")
print(daily_df.columns.tolist())

print(f"\n=== SAMPLE OF WORKING DAYS ===")
working_days = daily_df[daily_df['tipo'] == 'Trabalho']
print(working_days[['data', 'tipo', 'jornada_inicio', 'jornada_fim', 'jornada_normal', 'jornada_diaria']])

=== FINAL EXTRACTED INFORMATION ===
Company/Employer: None
Employee Name: ANDRE LUIS DE MORAES DA ROSA
Period: 21/05/2025 à 20/06/2025

=== SUMMARY OF DAILY TIMESHEET ===
Total days in timesheet: 31
Working days: 25
Rest days (DSR/Casa): 5
Holidays (Feriado): 1

=== TIMESHEET DATAFRAME COLUMNS ===
['data', 'tipo', 'jornada_inicio', 'jornada_fim', 'jornada_normal', 'jornada_diaria', 'interjornada', 'em_direcao', 'total_parado', 'sem_direcao', 'total_refeicao', 'total_repouso', 'hora_extra_diaria_diurna', 'hora_extra_diaria_noturna', 'hora_extra_diaria_total', 'hora_extra_dom_fer_diurna', 'hora_extra_dom_fer_noturna', 'hora_extra_dom_fer_total', 'hora_noturna']

=== SAMPLE OF WORKING DAYS ===
            data      tipo jornada_inicio jornada_fim jornada_normal jornada_diaria
0   21/05/25 qua  Trabalho          06:24       01:26          08:00          16:59
1   22/05/25 qui  Trabalho          10:35       23:42          08:00          11:58
2   23/05/25 sex  Trabalho          08:05       

In [42]:
# Labor Law Compliance Checks
from datetime import datetime, timedelta
import warnings

def time_to_minutes(time_str):
    """Convert time string HH:MM to minutes"""
    if pd.isna(time_str) or time_str is None or time_str == 'None':
        return 0
    try:
        hours, minutes = map(int, str(time_str).split(':'))
        return hours * 60 + minutes
    except:
        return 0

def minutes_to_time(minutes):
    """Convert minutes to HH:MM format"""
    hours = minutes // 60
    mins = minutes % 60
    return f"{hours:02d}:{mins:02d}"

def check_labor_compliance(daily_df):
    """Check labor law compliance for working days"""
    
    # Filter only working days
    working_days = daily_df[daily_df['tipo'] == 'Trabalho'].copy().reset_index(drop=True)
    
    compliance_issues = []
    
    # Variables for period totals
    total_working_minutes = 0
    total_meal_break_minutes = 0
    total_rest_hours = 0
    rest_periods_count = 0
    
    print("=== CRONOANÁLISE DA JORNADA DE TRABALHO ===\n")
    
    # Check 1: Daily working hours > 8 hours
    print("1. VALIDAÇÃO JORNADA DE TRABALHO DE 8 HORAS")
    print("-" * 50)
    
    for idx, row in working_days.iterrows():
        jornada_minutes = time_to_minutes(row['jornada_diaria'])
        jornada_hours = jornada_minutes / 60
        
        # Add to total working hours
        total_working_minutes += jornada_minutes
        
        if jornada_minutes > 480:  # 8 hours = 480 minutes
            excess_minutes = jornada_minutes - 480
            excess_time = minutes_to_time(excess_minutes)
            print(f"⚠️ {row['data']}: {row['jornada_diaria']} (EXCESO DE {excess_time} HORAS)")
            compliance_issues.append({
                'data': row['data'],
                'issue': 'Jornada diária excessiva',
                'details': f"Jornada trabalhada {row['jornada_diaria']}, excesso: {excess_time}"
            })
        else:
            print(f"✅ {row['data']}: {row['jornada_diaria']}")
    
    # Total for working hours section
    total_working_time = minutes_to_time(total_working_minutes)
    print(f"\n📊 TOTAL DE HORAS TRABALHADAS NO PERÍODO: {total_working_time}")
    
    # Check 2: Meal break >= 1 hour
    print(f"\n2. VALIDAÇÃO DO INTERVALO DE REFEIÇÃO (>= 1 hora)")
    print("-" * 50)
    
    for idx, row in working_days.iterrows():
        refeicao_minutes = time_to_minutes(row['total_refeicao'])
        
        # Add to total meal break minutes
        total_meal_break_minutes += refeicao_minutes
        
        if refeicao_minutes < 60:  # Less than 1 hour
            refeicao_time = minutes_to_time(refeicao_minutes) if refeicao_minutes > 0 else "Sem registro"
            print(f"⚠️ {row['data']}: {refeicao_time}")
            compliance_issues.append({
                'data': row['data'],
                'issue': 'Intervalo de refeição insuficiente',
                'details': f"Intervalo de refeição: {refeicao_time}, necessário: 01:00"
            })
        else:
            print(f"✅ {row['data']}: {row['total_refeicao']}")
    
    # Total for meal break section
    total_meal_break_time = minutes_to_time(total_meal_break_minutes)
    print(f"\n🍽️ TOTAL DE HORAS DE INTERVALO DE REFEIÇÃO NO PERÍODO: {total_meal_break_time}")
    
    # Check 3: Rest period between shifts >= 11 hours
    print(f"\n3. VALIDAÇÃO DO PERÍODO DE DESCANSO ENTRE TURNOS (>= 11 horas)")
    print("-" * 50)
    
    for i in range(len(working_days) - 1):
        current_day = working_days.iloc[i]
        next_day = working_days.iloc[i + 1]
        
        # Parse dates and times
        try:
            current_date_str = current_day['data'].split()[0]  # Get DD/MM/YY part
            next_date_str = next_day['data'].split()[0]

            current_fim = current_day['jornada_fim']
            next_inicio = next_day['jornada_inicio']

            if pd.notna(current_fim) and pd.notna(next_inicio):
                # Convert to datetime objects for calculation
                current_date = datetime.strptime(current_date_str, '%d/%m/%y')
                next_date = datetime.strptime(next_date_str, '%d/%m/%y')
                
                # Handle end time that goes to next day (e.g., 01:26 means 01:26 next day)
                fim_hour, fim_min = map(int, str(current_fim).split(':'))
                inicio_hour, inicio_min = map(int, str(next_inicio).split(':'))
                
                # If fim time is small (like 01:26), it likely means next day
                if fim_hour < 6:  # Assuming work doesn't normally end before 6 AM
                    fim_datetime = current_date + timedelta(days=1, hours=fim_hour, minutes=fim_min)
                else:
                    fim_datetime = current_date + timedelta(hours=fim_hour, minutes=fim_min)
                
                inicio_datetime = next_date + timedelta(hours=inicio_hour, minutes=inicio_min)
                
                # Calculate rest period
                rest_period = inicio_datetime - fim_datetime
                rest_hours = rest_period.total_seconds() / 3600
                
                # Add to total rest hours
                total_rest_hours += rest_hours
                rest_periods_count += 1
                
                if rest_hours < 11:
                    rest_time_str = f"{int(rest_hours):02d}:{int((rest_hours % 1) * 60):02d}"
                    faltando_horas = 11 - rest_hours
                    faltando_horas_str = f"{int(faltando_horas):02d}:{int((faltando_horas % 1) * 60):02d}"
                    print(f"⚠️ {current_day['data']} ({current_fim}) → {next_day['data']} ({next_inicio}): DESCANSO {rest_time_str} - FALTARAM {faltando_horas_str}")
                    compliance_issues.append({
                        'data': f"{current_day['data']} → {next_day['data']}",
                        'issue': 'Período de descanso insuficiente',
                        'details': f"Período de descanso: {rest_time_str}, necessário: 11:00"
                    })
                else:
                    rest_time_str = f"{int(rest_hours):02d}:{int((rest_hours % 1) * 60):02d}"
                    print(f"✅ {current_day['data']} ({current_fim}) → {next_day['data']} ({next_inicio}): DESCANSO {rest_time_str}")
                    
        except Exception as e:
            print(f"⚠️  Erro calculando período de descanso entre {current_day['data']} e {next_day['data']}: {str(e)}")

    # Total for rest period section
    if rest_periods_count > 0:
        average_rest_hours = total_rest_hours / rest_periods_count
        average_rest_time = f"{int(average_rest_hours):02d}:{int((average_rest_hours % 1) * 60):02d}"
        print(f"\n😴 TEMPO MÉDIO DE DESCANSO ENTRE TURNOS: {average_rest_time}")
        print(f"🔄 NÚMERO DE PERÍODOS DE DESCANSO ANALISADOS: {rest_periods_count}")

    # Summary
    print(f"\n=== RESUMO DE CONFORMIDADE ===")
    print(f"Total de dias trabalhados analisados: {len(working_days)}")
    print(f"Total de problemas de conformidade encontrados: {len(compliance_issues)}")
    
    print(f"\n=== TOTAIS CONSOLIDADOS DO PERÍODO ===")
    print(f"📊 Total de horas trabalhadas: {total_working_time}")
    print(f"🍽️  Total de horas de intervalo de refeição: {total_meal_break_time}")
    if rest_periods_count > 0:
        print(f"😴 Tempo médio de descanso entre turnos: {average_rest_time}")
        print(f"🔄 Número de períodos de descanso analisados: {rest_periods_count}")

    if compliance_issues:
        print(f"\n=== PROBLEMAS DETALHADOS ===")
        
        # Group issues by category
        jornada_issues = [issue for issue in compliance_issues if 'Jornada diária excessiva' in issue['issue']]
        refeicao_issues = [issue for issue in compliance_issues if 'Intervalo de refeição' in issue['issue']]
        descanso_issues = [issue for issue in compliance_issues if 'Período de descanso' in issue['issue']]
        
        # Display Jornada issues
        if jornada_issues:
            print(f"\n🕐 JORNADAS DIÁRIAS EXCESSIVAS:")
            print("-" * 40)
            for issue in jornada_issues:
                print(f"   📅 {issue['data']}")
                print(f"      {issue['details']}")
        
        # Display Meal break issues
        if refeicao_issues:
            print(f"\n🍽️  INTERVALOS DE REFEIÇÃO INSUFICIENTES:")
            print("-" * 40)
            for issue in refeicao_issues:
                print(f"   📅 {issue['data']}")
                print(f"      {issue['details']}")
        
        # Display Rest period issues
        if descanso_issues:
            print(f"\n😴 PERÍODOS DE DESCANSO INSUFICIENTES:")
            print("-" * 40)
            for issue in descanso_issues:
                print(f"   📅 {issue['data']}")
                print(f"      {issue['details']}")
    else:
        print("🎉 Nenhum problema de conformidade encontrado!")
    
    return compliance_issues

# Run the compliance check
compliance_issues = check_labor_compliance(daily_df)

=== CRONOANÁLISE DA JORNADA DE TRABALHO ===

1. VALIDAÇÃO JORNADA DE TRABALHO DE 8 HORAS
--------------------------------------------------
⚠️ 21/05/25 qua: 16:59 (EXCESO DE 08:59 HORAS)
⚠️ 22/05/25 qui: 11:58 (EXCESO DE 03:58 HORAS)
⚠️ 23/05/25 sex: 10:56 (EXCESO DE 02:56 HORAS)
⚠️ 24/05/25 sáb: 08:33 (EXCESO DE 00:33 HORAS)
⚠️ 25/05/25 dom: 10:08 (EXCESO DE 02:08 HORAS)
⚠️ 26/05/25 seg: 14:21 (EXCESO DE 06:21 HORAS)
⚠️ 27/05/25 ter: 10:31 (EXCESO DE 02:31 HORAS)
⚠️ 28/05/25 qua: 14:33 (EXCESO DE 06:33 HORAS)
⚠️ 29/05/25 qui: 08:17 (EXCESO DE 00:17 HORAS)
⚠️ 30/05/25 sex: 12:18 (EXCESO DE 04:18 HORAS)
⚠️ 31/05/25 sáb: 09:39 (EXCESO DE 01:39 HORAS)
⚠️ 01/06/25 dom: 08:53 (EXCESO DE 00:53 HORAS)
⚠️ 02/06/25 seg: 13:51 (EXCESO DE 05:51 HORAS)
⚠️ 03/06/25 ter: 11:19 (EXCESO DE 03:19 HORAS)
⚠️ 04/06/25 qua: 08:48 (EXCESO DE 00:48 HORAS)
⚠️ 05/06/25 qui: 10:10 (EXCESO DE 02:10 HORAS)
✅ 06/06/25 sex: 01:15
⚠️ 09/06/25 seg: 13:22 (EXCESO DE 05:22 HORAS)
⚠️ 10/06/25 ter: 09:14 (EXCESO DE 01:14

In [43]:
# Instalar dependências para o dashboard web
!pip install streamlit plotly

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: C:\Python313\python.exe -m pip install --upgrade pip


In [44]:
# Criar arquivo dashboard_cronanalise.py para o Streamlit
dashboard_code = '''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import json

# Configuração da página
st.set_page_config(
    page_title="Dashboard - Cronoanálise da Jornada de Trabalho",
    page_icon="⏰",
    layout="wide"
)

# Título principal
st.title("📊 Dashboard - Cronoanálise da Jornada de Trabalho")
st.markdown("---")

# Funções auxiliares
def time_to_minutes(time_str):
    """Convert time string HH:MM to minutes"""
    if pd.isna(time_str) or time_str is None or time_str == 'None':
        return 0
    try:
        hours, minutes = map(int, str(time_str).split(':'))
        return hours * 60 + minutes
    except:
        return 0

def minutes_to_time(minutes):
    """Convert minutes to HH:MM format"""
    hours = minutes // 60
    mins = minutes % 60
    return f"{hours:02d}:{mins:02d}"

# Simulação dos dados (substitua pelos seus dados reais)
# Para usar seus dados reais, carregue do arquivo ou banco de dados
sample_data = {
    'data': ['01/03/23 Qua', '02/03/23 Qui', '03/03/23 Sex', '04/03/23 Sab', '05/03/23 Dom',
             '06/03/23 Seg', '07/03/23 Ter', '08/03/23 Qua', '09/03/23 Qui', '10/03/23 Sex'],
    'tipo': ['Trabalho', 'Trabalho', 'Trabalho', 'DSR/Casa', 'DSR/Casa', 
             'Trabalho', 'Trabalho', 'Trabalho', 'Trabalho', 'Trabalho'],
    'jornada_inicio': ['18:00', '18:00', '18:00', None, None, 
                       '18:00', '18:00', '18:00', '18:00', '18:00'],
    'jornada_fim': ['01:26', '02:26', '02:26', None, None,
                    '02:26', '02:26', '02:26', '02:26', '02:26'],
    'jornada_diaria': ['07:26', '08:26', '08:26', None, None,
                       '08:26', '08:26', '08:26', '08:26', '08:26'],
    'total_refeicao': ['01:00', '01:00', '01:00', None, None,
                       '01:00', '01:00', '01:00', '01:00', '01:00']
}

df = pd.DataFrame(sample_data)

# Informações do funcionário (substitua pelos dados reais)
employee_info = {
    'nome': 'ANDRE LUIS DE MORAES DA ROSA',
    'periodo': '03/2023',
    'empresa': 'Empresa Exemplo'
}

# Sidebar com informações
st.sidebar.header("📋 Informações do Relatório")
st.sidebar.write(f"**Funcionário:** {employee_info['nome']}")
st.sidebar.write(f"**Período:** {employee_info['periodo']}")
st.sidebar.write(f"**Empresa:** {employee_info['empresa']}")

# Análise dos dados de trabalho
working_days = df[df['tipo'] == 'Trabalho'].copy()

# Métricas principais
col1, col2, col3, col4 = st.columns(4)

total_days = len(df)
working_days_count = len(working_days)
rest_days = len(df[df['tipo'] == 'DSR/Casa'])

# Calcular totais
total_working_minutes = sum(time_to_minutes(jornada) for jornada in working_days['jornada_diaria'] if pd.notna(jornada))
total_working_time = minutes_to_time(total_working_minutes)

total_meal_minutes = sum(time_to_minutes(refeicao) for refeicao in working_days['total_refeicao'] if pd.notna(refeicao))
total_meal_time = minutes_to_time(total_meal_minutes)

with col1:
    st.metric("📅 Total de Dias", total_days)

with col2:
    st.metric("🏢 Dias Trabalhados", working_days_count)

with col3:
    st.metric("⏰ Total Horas Trabalhadas", total_working_time)

with col4:
    st.metric("🍽️ Total Intervalos Refeição", total_meal_time)

st.markdown("---")

# Gráficos
col1, col2 = st.columns(2)

# Gráfico 1: Distribuição de tipos de dia
with col1:
    st.subheader("📊 Distribuição dos Tipos de Dia")
    tipo_counts = df['tipo'].value_counts()
    fig_pie = px.pie(values=tipo_counts.values, names=tipo_counts.index, 
                     color_discrete_sequence=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    fig_pie.update_traces(textposition='inside', textinfo='percent+label')
    st.plotly_chart(fig_pie, use_container_width=True)

# Gráfico 2: Horas trabalhadas por dia
with col2:
    st.subheader("⏱️ Jornada Diária de Trabalho")
    working_days_chart = working_days.copy()
    working_days_chart['horas_numericas'] = working_days_chart['jornada_diaria'].apply(
        lambda x: time_to_minutes(x) / 60 if pd.notna(x) else 0
    )
    working_days_chart['data_simples'] = working_days_chart['data'].str[:8]
    
    fig_bar = px.bar(working_days_chart, x='data_simples', y='horas_numericas',
                     title="Horas por Dia",
                     color='horas_numericas',
                     color_continuous_scale='RdYlGn_r')
    fig_bar.add_hline(y=8, line_dash="dash", line_color="red", 
                      annotation_text="Limite 8h")
    fig_bar.update_layout(showlegend=False)
    st.plotly_chart(fig_bar, use_container_width=True)

# Análise de conformidade
st.markdown("---")
st.subheader("⚖️ Análise de Conformidade Trabalhista")

# Verificar jornadas excessivas
excessive_days = []
meal_issues = []

for idx, row in working_days.iterrows():
    jornada_minutes = time_to_minutes(row['jornada_diaria'])
    if jornada_minutes > 480:  # > 8 horas
        excess_time = minutes_to_time(jornada_minutes - 480)
        excessive_days.append({
            'data': row['data'],
            'jornada': row['jornada_diaria'],
            'excesso': excess_time
        })
    
    # Verificar intervalo de refeição
    meal_minutes = time_to_minutes(row['total_refeicao'])
    if meal_minutes < 60:  # < 1 hora
        meal_issues.append({
            'data': row['data'],
            'intervalo': row['total_refeicao'] if pd.notna(row['total_refeicao']) else "Sem registro"
        })

# Exibir problemas encontrados
col1, col2 = st.columns(2)

with col1:
    st.markdown("#### 🕐 Jornadas Excessivas")
    if excessive_days:
        for day in excessive_days:
            st.error(f"📅 {day['data']}: {day['jornada']} (Excesso: {day['excesso']})")
    else:
        st.success("✅ Nenhuma jornada excessiva encontrada!")

with col2:
    st.markdown("#### 🍽️ Problemas nos Intervalos")
    if meal_issues:
        for issue in meal_issues:
            st.warning(f"📅 {issue['data']}: {issue['intervalo']}")
    else:
        st.success("✅ Todos os intervalos estão adequados!")

# Resumo final
st.markdown("---")
st.subheader("📋 Resumo Executivo")

total_issues = len(excessive_days) + len(meal_issues)

if total_issues == 0:
    st.success("🎉 **PARABÉNS!** Nenhum problema de conformidade encontrado!")
    st.balloons()
else:
    st.error(f"⚠️ **ATENÇÃO:** {total_issues} problema(s) de conformidade encontrado(s)")
    
    with st.expander("Ver detalhes dos problemas"):
        if excessive_days:
            st.markdown("**Jornadas Excessivas:**")
            for day in excessive_days:
                st.write(f"• {day['data']}: {day['jornada']} (Excesso: {day['excesso']})")
        
        if meal_issues:
            st.markdown("**Intervalos Insuficientes:**")
            for issue in meal_issues:
                st.write(f"• {issue['data']}: {issue['intervalo']}")

# Footer
st.markdown("---")
st.markdown("*Dashboard gerado automaticamente pela análise de cronoanálise da jornada de trabalho*")
'''

# Salvar o arquivo
with open('dashboard_cronanalise.py', 'w', encoding='utf-8') as f:
    f.write(dashboard_code)

print("✅ Arquivo 'dashboard_cronanalise.py' criado com sucesso!")
print("📁 Localização: pasta atual do projeto")
print("\n🚀 Para executar o dashboard:")
print("1. Abra um terminal")
print("2. Execute: streamlit run dashboard_cronanalise.py")
print("3. O dashboard abrirá automaticamente no seu navegador")

✅ Arquivo 'dashboard_cronanalise.py' criado com sucesso!
📁 Localização: pasta atual do projeto

🚀 Para executar o dashboard:
1. Abra um terminal
2. Execute: streamlit run dashboard_cronanalise.py
3. O dashboard abrirá automaticamente no seu navegador


In [45]:
# Criar dashboard personalizado com os dados reais
dashboard_real_data_code = f'''
import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime, timedelta
import json

# Configuração da página
st.set_page_config(
    page_title="Dashboard - Cronoanálise da Jornada de Trabalho - {employee_name}",
    page_icon="⏰",
    layout="wide"
)

# Título principal
st.title("📊 Dashboard - Cronoanálise da Jornada de Trabalho")
st.subheader(f"Funcionário: {employee_name}")
st.markdown("---")

# Funções auxiliares
def time_to_minutes(time_str):
    """Convert time string HH:MM to minutes"""
    if pd.isna(time_str) or time_str is None or time_str == 'None':
        return 0
    try:
        hours, minutes = map(int, str(time_str).split(':'))
        return hours * 60 + minutes
    except:
        return 0

def minutes_to_time(minutes):
    """Convert minutes to HH:MM format"""
    hours = minutes // 60
    mins = minutes % 60
    return f"{{hours:02d}}:{{mins:02d}}"

# Dados reais do funcionário
data_dict = {daily_df.to_dict('list')}

df = pd.DataFrame(data_dict)

# Informações do funcionário
employee_info = {{
    'nome': '{employee_name}',
    'periodo': '{period}',
    'empresa': '{company_name if company_name else "Não identificada"}'
}}

# Sidebar com informações
st.sidebar.header("📋 Informações do Relatório")
st.sidebar.write(f"**Funcionário:** {{employee_info['nome']}}")
st.sidebar.write(f"**Período:** {{employee_info['periodo']}}")
st.sidebar.write(f"**Empresa:** {{employee_info['empresa']}}")

# Análise dos dados de trabalho
working_days = df[df['tipo'] == 'Trabalho'].copy()
rest_days = df[df['tipo'] == 'DSR/Casa'].copy()
holiday_days = df[df['tipo'] == 'Feriado'].copy()

# Métricas principais
col1, col2, col3, col4 = st.columns(4)

total_days = len(df)
working_days_count = len(working_days)
rest_days_count = len(rest_days)

# Calcular totais
total_working_minutes = sum(time_to_minutes(jornada) for jornada in working_days['jornada_diaria'] if pd.notna(jornada))
total_working_time = minutes_to_time(total_working_minutes)

total_meal_minutes = sum(time_to_minutes(refeicao) for refeicao in working_days['total_refeicao'] if pd.notna(refeicao))
total_meal_time = minutes_to_time(total_meal_minutes)

with col1:
    st.metric("📅 Total de Dias", total_days)

with col2:
    st.metric("🏢 Dias Trabalhados", working_days_count)

with col3:
    st.metric("⏰ Total Horas Trabalhadas", total_working_time)

with col4:
    st.metric("🍽️ Total Intervalos Refeição", total_meal_time)

# Segunda linha de métricas
col1, col2, col3, col4 = st.columns(4)

with col1:
    st.metric("🏠 Dias de Descanso", rest_days_count)

with col2:
    st.metric("🎉 Feriados", len(holiday_days))

with col3:
    # Calcular horas extras
    total_extra_minutes = sum(time_to_minutes(extra) for extra in working_days['hora_extra_diaria_total'] if pd.notna(extra))
    total_extra_time = minutes_to_time(total_extra_minutes)
    st.metric("⏰ Total Horas Extras", total_extra_time)

with col4:
    # Calcular horas noturnas
    total_night_minutes = sum(time_to_minutes(night) for night in working_days['hora_noturna'] if pd.notna(night))
    total_night_time = minutes_to_time(total_night_minutes)
    st.metric("🌙 Total Horas Noturnas", total_night_time)

st.markdown("---")

# Gráficos
col1, col2 = st.columns(2)

# Gráfico 1: Distribuição de tipos de dia
with col1:
    st.subheader("📊 Distribuição dos Tipos de Dia")
    tipo_counts = df['tipo'].value_counts()
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA726', '#AB47BC']
    fig_pie = px.pie(values=tipo_counts.values, names=tipo_counts.index, 
                     color_discrete_sequence=colors)
    fig_pie.update_traces(textposition='inside', textinfo='percent+label')
    st.plotly_chart(fig_pie, use_container_width=True)

# Gráfico 2: Horas trabalhadas por dia
with col2:
    st.subheader("⏱️ Jornada Diária de Trabalho")
    working_days_chart = working_days.copy()
    working_days_chart['horas_numericas'] = working_days_chart['jornada_diaria'].apply(
        lambda x: time_to_minutes(x) / 60 if pd.notna(x) else 0
    )
    working_days_chart['data_simples'] = working_days_chart['data'].str[:8]
    
    fig_bar = px.bar(working_days_chart, x='data_simples', y='horas_numericas',
                     title="Horas por Dia",
                     color='horas_numericas',
                     color_continuous_scale='RdYlGn_r')
    fig_bar.add_hline(y=8, line_dash="dash", line_color="red", 
                      annotation_text="Limite Legal 8h")
    fig_bar.update_layout(showlegend=False, xaxis_title="Data", yaxis_title="Horas")
    st.plotly_chart(fig_bar, use_container_width=True)

# Gráfico 3: Evolução das horas extras
st.subheader("📈 Evolução das Horas Extras e Noturnas")
col1, col2 = st.columns(2)

with col1:
    # Horas extras por dia
    extras_chart = working_days.copy()
    extras_chart['extra_numericas'] = extras_chart['hora_extra_diaria_total'].apply(
        lambda x: time_to_minutes(x) / 60 if pd.notna(x) else 0
    )
    extras_chart['data_simples'] = extras_chart['data'].str[:8]
    
    fig_extra = px.line(extras_chart, x='data_simples', y='extra_numericas',
                       title="Horas Extras por Dia",
                       markers=True)
    fig_extra.update_layout(xaxis_title="Data", yaxis_title="Horas Extras")
    st.plotly_chart(fig_extra, use_container_width=True)

with col2:
    # Horas noturnas por dia
    night_chart = working_days.copy()
    night_chart['noturnas_numericas'] = night_chart['hora_noturna'].apply(
        lambda x: time_to_minutes(x) / 60 if pd.notna(x) else 0
    )
    night_chart['data_simples'] = night_chart['data'].str[:8]
    
    fig_night = px.bar(night_chart, x='data_simples', y='noturnas_numericas',
                      title="Horas Noturnas por Dia",
                      color_discrete_sequence=['#2E86AB'])
    fig_night.update_layout(xaxis_title="Data", yaxis_title="Horas Noturnas")
    st.plotly_chart(fig_night, use_container_width=True)

# Análise de conformidade
st.markdown("---")
st.subheader("⚖️ Análise de Conformidade Trabalhista")

# Verificar jornadas excessivas
excessive_days = []
meal_issues = []
rest_issues = []

for idx, row in working_days.iterrows():
    jornada_minutes = time_to_minutes(row['jornada_diaria'])
    if jornada_minutes > 480:  # > 8 horas
        excess_time = minutes_to_time(jornada_minutes - 480)
        excessive_days.append({{
            'data': row['data'],
            'jornada': row['jornada_diaria'],
            'excesso': excess_time
        }})
    
    # Verificar intervalo de refeição
    meal_minutes = time_to_minutes(row['total_refeicao'])
    if meal_minutes < 60:  # < 1 hora
        meal_issues.append({{
            'data': row['data'],
            'intervalo': row['total_refeicao'] if pd.notna(row['total_refeicao']) else "Sem registro"
        }})

# Problemas de conformidade conhecidos
compliance_issues_list = {compliance_issues}

# Separar por tipo
jornada_issues = [issue for issue in compliance_issues_list if 'Jornada diária excessiva' in issue['issue']]
refeicao_issues = [issue for issue in compliance_issues_list if 'Intervalo de refeição' in issue['issue']]
descanso_issues = [issue for issue in compliance_issues_list if 'Período de descanso' in issue['issue']]

# Exibir problemas encontrados
col1, col2, col3 = st.columns(3)

with col1:
    st.markdown("#### 🕐 Jornadas Excessivas")
    if jornada_issues:
        for issue in jornada_issues:
            st.error(f"📅 {{issue['data']}}")
            st.caption(issue['details'])
    else:
        st.success("✅ Nenhuma jornada excessiva!")

with col2:
    st.markdown("#### 🍽️ Intervalos de Refeição")
    if refeicao_issues:
        for issue in refeicao_issues:
            st.warning(f"📅 {{issue['data']}}")
            st.caption(issue['details'])
    else:
        st.success("✅ Intervalos adequados!")

with col3:
    st.markdown("#### 😴 Períodos de Descanso")
    if descanso_issues:
        for issue in descanso_issues:
            st.error(f"📅 {{issue['data']}}")
            st.caption(issue['details'])
    else:
        st.success("✅ Descansos adequados!")

# Tabela detalhada
st.markdown("---")
st.subheader("📋 Dados Detalhados")

# Filtro de tipo de dia
tipo_filter = st.selectbox("Filtrar por tipo de dia:", 
                          options=['Todos'] + list(df['tipo'].unique()))

if tipo_filter == 'Todos':
    filtered_df = df
else:
    filtered_df = df[df['tipo'] == tipo_filter]

# Exibir tabela
st.dataframe(filtered_df, use_container_width=True)

# Resumo final
st.markdown("---")
st.subheader("📋 Resumo Executivo")

total_issues = len(compliance_issues_list)

if total_issues == 0:
    st.success("🎉 **PARABÉNS!** Nenhum problema de conformidade encontrado!")
    st.balloons()
else:
    st.error(f"⚠️ **ATENÇÃO:** {{total_issues}} problema(s) de conformidade encontrado(s)")
    
    # Estatísticas de problemas
    col1, col2, col3 = st.columns(3)
    
    with col1:
        st.metric("🕐 Jornadas Excessivas", len(jornada_issues))
    
    with col2:
        st.metric("🍽️ Intervalos Insuficientes", len(refeicao_issues))
    
    with col3:
        st.metric("😴 Descansos Insuficientes", len(descanso_issues))
    
    with st.expander("Ver todos os problemas detalhados"):
        for issue in compliance_issues_list:
            if 'Jornada diária excessiva' in issue['issue']:
                st.error(f"🕐 {{issue['data']}}: {{issue['details']}}")
            elif 'Intervalo de refeição' in issue['issue']:
                st.warning(f"🍽️ {{issue['data']}}: {{issue['details']}}")
            elif 'Período de descanso' in issue['issue']:
                st.info(f"😴 {{issue['data']}}: {{issue['details']}}")

# Footer
st.markdown("---")
st.markdown("*Dashboard gerado automaticamente pela análise de cronoanálise da jornada de trabalho*")
st.markdown(f"*Dados extraídos de: {{pdf_path}}*")
'''

# Salvar o arquivo personalizado
with open('dashboard_cronanalise_real.py', 'w', encoding='utf-8') as f:
    f.write(dashboard_real_data_code)

print("✅ Dashboard personalizado criado: 'dashboard_cronanalise_real.py'")
print("📊 Este dashboard usa os dados reais extraídos do PDF")
print(f"👤 Funcionário: {employee_name}")
print(f"📅 Período: {period}")
print(f"🏢 Empresa: {company_name if company_name else 'Não identificada'}")
print(f"📋 Total de problemas de conformidade: {len(compliance_issues)}")
print("\n🚀 Para executar o dashboard:")
print("1. Execute a célula anterior para instalar as dependências")
print("2. Abra um terminal")
print("3. Execute: streamlit run dashboard_cronanalise_real.py")
print("4. O dashboard abrirá automaticamente no seu navegador em http://localhost:8501")

✅ Dashboard personalizado criado: 'dashboard_cronanalise_real.py'
📊 Este dashboard usa os dados reais extraídos do PDF
👤 Funcionário: ANDRE LUIS DE MORAES DA ROSA
📅 Período: 21/05/2025 à 20/06/2025
🏢 Empresa: Não identificada
📋 Total de problemas de conformidade: 34

🚀 Para executar o dashboard:
1. Execute a célula anterior para instalar as dependências
2. Abra um terminal
3. Execute: streamlit run dashboard_cronanalise_real.py
4. O dashboard abrirá automaticamente no seu navegador em http://localhost:8501
